<a href="https://colab.research.google.com/github/JCARNEIROX/IA367-Aprendizado-Reforco/blob/main/Lista2_Ex11_RA239738_RA256389.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**IA368FF - Aprendizado por Reforço**  
1º Semestre de 2024  
Prof. Denis Fantinato  

In [1]:
from copy import deepcopy
import numpy as np

from collections import defaultdict

# Criando o MDP

**Grid World 4x3**  
Vamos criar um Grid World 4x3 como um MDP.  

O MDP é definido por:  
                        MDP = (𝑆, 𝐴, 𝑅, ℙ, 𝛾)
com
- 𝑆: conjunto de possíveis estados.
- 𝐴: conjunto de ações.
- 𝑅 ∶ 𝑆 → ℝ: mapa de recompensa para cada estado.
- ℙ: probabilidade de transição de um estado para outro dada uma ação.
- 𝛾: fator de desconto. Um número entre 0 e 1.  
  
Neste mesmo bloco, definimos as probabilidades de transição de estados. Note que o agente tem 80\% de chance de seguir na direção da ação escolhida e 10\% de chance para cada direção perpendicular.

In [2]:
def createMDP():

    '''
    definição do ambiente
    '''

    S       = [(i,j) for i in range(1,5)
                     for j in range(1,4) if (i,j) != (2,2)]

    goals   = [(4,3), (4,2)]
    actions = ["UP", "DOWN", "LEFT", "RIGHT"]
    A       = {s : actions
               for s in S }

    R        = {s : -0.04 for s in S}
    R[(4,3)] =  1
    R[(4,2)] = -1

    P        = { (s,a) : pvals(s, a, S) for s in S for a in A[s] }

    gamma    = .9

    return (S,A,R,P,gamma)


def move(s, a, S):
    i, j = s
    if a == "UP":
        sp = (i, j+1)
    elif a == "DOWN":
        sp = (i, j-1)
    elif a == "LEFT":
        sp = (i-1, j)
    elif a == "RIGHT":
        sp = (i+1, j)
    elif a is None:
        return s

    if sp in S:
        return sp

    return s

def succ(a):
    return {"UP": "RIGHT", "DOWN": "LEFT", "RIGHT": "DOWN", "LEFT": "UP", None : None}[a]

def pred(a):
    return {"UP": "LEFT", "DOWN": "RIGHT", "RIGHT": "UP", "LEFT": "DOWN", None : None}[a]

def pvals(s, a, S):
    return [(0.8, move(s, a, S)), (0.1, move(s, succ(a), S)), (0.1, move(s, pred(a), S))]

Para confirmar se entendeu esse bloco:    
- Onde a recompensa pode ser ajustada?

  `R:` A recompensa pode ser ajustada trocando o valor "-0.04" na linha de código 15.
- Como ficaria a lista `S`? Escreva os estados na ordem correta.

  `R:` S = [(1, 1),(1, 2), (1, 3), (2, 1), (2, 3), (3, 1), (3, 2), (3, 3), (4, 1), (4, 2), (4, 3)]

- O que há em:
  - `A[(3,2)]`?

    `R:` A[(3,2)] = ['UP', 'DOWN', 'LEFT', 'RIGHT']

  - `A[(2,2)]`?

    `R:` A[(2,2)] = State (2,2) does not exist

  - `A[(4,2)]`?

    `R:` A[(4,2)] = ['UP', 'DOWN', 'LEFT', 'RIGHT']
- Qual a saída para `P[((3,3),"RIGHT")] = pvals((3,3), "RIGHT", S)`?

    `R:`P[((3,3),"RIGHT")] = [(0.8, (4, 3)), (0.1, (3, 2)), (0.1, (3, 3))]    
- O que acontece se o resultado for um estado fora do grid?

    `R:` O Agente não se movimenta e permanece no estado atual.


# Algoritmo Q-Learning


Q-Learning é um algoritmo de diferença temporal que aprende uma função `Q(s, a)` que indica a qualidade em escolher a ação `a` no estado `s`.

In [7]:
def qlearn(percept, state, goals, A, gamma, alpha, f):
    sn, rn        = percept
    s, a, r, N, Q = state

    # se for estado final, usa a recompensa
    if sn in goals:
        Q[(sn, None)] = rn

    # se s nao for nulo, atualiza Q
    if s is not None:
        N[(s,a)] = N[(s,a)] + 1
        maxQ     = max(Q[(sn,ai)] for ai in A[sn])
        Q[(s,a)] = Q[(s,a)] + alpha * (r + gamma*maxQ - Q[(s,a)])

    # avalia o proximo estado e a acao que deve ser tomada
    if s in goals:
        state = (None, None, 0, N, Q)
    else:
        qvals    = [(f(Q[(sn,ai)], N[(sn,ai)]), ai) for ai in A[sn]]
        an       = max(qvals, key=lambda x: x[0])[1]
        state    = (sn, an, rn, N, Q)
    return an, state

def createF(nMax, rPlus):
    def f(q,n):
        if n < nMax:
            return rPlus
        return q
    return f

Para confirmar se entendeu esse bloco:
- Se `nMax = 10`, `rPlus = 2`, o que a função `f(3.4,4)` e `f(3.8, 11)` retornam?

    `R:` f(3.4,4) = 2  pois n < nMax retornando rPlus =2, e f(3.4,8)=3.8 pois n > nMax retornando q=3.8 . 

In [ ]:
def runQ(mdp, nTrials, alpha, nMax, rPlus):
    S, A, R, P, gamma = mdp

    s0        = (1,1)
    goals     = [(4,3), (4,2)]

    Q = defaultdict(float)
    N = defaultdict(int)
    f = createF(nMax, rPlus)

    for t in range(nTrials):
        # Estado inicial nulo
        state = None, None, 0, N, Q
        s = s0
        percept = (s, R[s])
        while state[0] not in goals:
            a, state = qlearn(percept, state, goals, A, gamma, alpha, f)
            if s not in goals:
                s   = performQAction(s, a, P)
            percept = (s, R[s])

    # Transforma Q-values em politica
    s, a, r, N, Q = state
    pi            = {}
    for s in S:
        qvals    = [(Q[(s,a)], a) for a in A[s]]
        maxQ, an = max(qvals, key=lambda x: x[0])
        pi[s] = an #.append( (s,an, maxQ) )
    return pi

def performQAction(s, a, P):
    ps     = P[(s, a)]
    probs  = list(map(lambda x: x[0], ps))
    states = list(map(lambda x: x[1], ps))
    idx    = np.random.choice(len(states), p=probs)

    return states[idx]

O bloco acima implementa a execução do algoritmo Q-learning.

# Executando o Algoritmo Q-Learning

Função principal.  
Note que a política está sendo estimada pelo método de Iteração-de-Política.

In [76]:
def main():
    mdp = createMDP()
    piQ = runQ(mdp, 100, 0.1, 100, 2)

    print("Q-Learning: \n")
    for j in range(3,0,-1):
        for i in range(1,5):
            if (i,j) in piQ:
                v = piQ[(i,j)]
                print(f"|  {v}  |", end="")
            else:
                print("|       |", end="")
        print("")

main()

Q-Learning: 

|  DOWN  ||  UP  ||  RIGHT  ||  UP  |
|  UP  ||       ||  LEFT  ||  UP  |
|  RIGHT  ||  RIGHT  ||  DOWN  ||  DOWN  |


In [77]:
s = (1,1)
goals = [(4,3), (4,2)]
trajetoria = [s]

while s not in goals:
    a = piQ[s]
    s = performQAction(s, a, P)
    trajetoria.append(s)

print("Caminho percorrido:")
print(trajetoria)

Caminho percorrido:
[(1, 1), (2, 1), (3, 1), (4, 1), (4, 2)]
